# Map the sweep IDs to names using the wanbd API

In [ ]:
import wandb
import pandas as pd

# === Settings ===
ENTITY = ""        
PROJECT = "graph-uncertainty"      
INPUT_CSV = "/Users/graph-uncertainty/wandb_export_2025-11-18T20_57_43.211+01_00.csv"
OUTPUT_CSV = "./wandb_export_final.csv"

# === Load CSV ===
df = pd.read_csv(INPUT_CSV)

# Get unique sweep IDs from the CSV
sweep_ids = df["Sweep"].dropna().unique()

# === Fetch sweep names from API ===
api = wandb.Api()
id_to_name = {}

for sid in sweep_ids:
    try:
        sweep = api.sweep(f"{ENTITY}/{PROJECT}/{sid}")
        id_to_name[sid] = sweep.name or sid  # fallback to ID if no name
    except Exception as e:
        print(f"⚠️ Could not fetch sweep {sid}: {e}")
        id_to_name[sid] = sid  # fallback to ID

# === Replace IDs with names ===
df["Sweep"] = df["Sweep"].map(id_to_name)

# === Save new CSV ===
df.to_csv(OUTPUT_CSV, index=False)

print(f"✅ Updated CSV saved as {OUTPUT_CSV}")


# Differences between different conv layers

In [ ]:
import pandas as pd

df = pd.read_csv("./wandb_export_final.csv")

In [ ]:
# take the df where the Sweep is something ending with "credal_LJ"
df = df[df["Sweep"].str.endswith("credal_LJ", na=False)]
# add the column "dataset" that is the part of the Sweep before the first underscore
# avoid SettingWithCopyWarning by working on a copy
df = df.copy()
df["dataset"] = df["Sweep"].str.split("_").str[0]

In [ ]:
# take top 3 rows by 'test_auroc_EU' for each (dataset, gnn_type), then compute means
best_results = (
	df.sort_values(['dataset', 'gnn_type', 'test_auroc_EU'], ascending=[True, True, False])
	  .groupby(['dataset', 'gnn_type'], as_index=False)
	  .head(3)
	  .reset_index(drop=True)
)

# mean across all datasets per gnn_type (using the selected top-3 per dataset)
mean_results = best_results.groupby('gnn_type', as_index=False)['test_auroc_EU'].mean()

# (optional) mean per dataset and gnn_type (i.e. mean of the top-3 within each dataset)
mean_per_dataset = best_results.groupby(['dataset', 'gnn_type'], as_index=False)['test_auroc_EU'].mean()

In [ ]:
mean_per_dataset

In [ ]:
# compute the percentage difference for the two different gnn_types per dataset
mean_per_dataset_pivot = mean_per_dataset.pivot(index='dataset', columns='gnn_type', values='test_auroc_EU').reset_index()
mean_per_dataset_pivot['percentage_diff'] = (
	(mean_per_dataset_pivot['GCN'] - mean_per_dataset_pivot['SAGE']) / mean_per_dataset_pivot['SAGE']
) * 100

In [ ]:
mean_per_dataset_pivot

# differences betweeen num layers

In [ ]:
# similar study as above, but with the number of layers instead of gnn_type
import pandas as pd

df = pd.read_csv("./wandb_export_final.csv")

# take the df where the Sweep is something ending with "credal"
df = df[df["Sweep"].str.endswith("credal", na=False)]
# add the column "dataset" that is the part of the Sweep before the first underscore
# avoid SettingWithCopyWarning by working on a copy
df = df.copy()
df["dataset"] = df["Sweep"].str.split("_").str[0]

# filter num_layers to be only 2 or 3
df = df[df["num_layers"].isin([2, 3])]

In [ ]:
# take top 3 rows by 'test_auroc_EU' for each (dataset, num_layers), then compute means
best_results = (
	df.sort_values(['dataset', 'num_layers', 'test_auroc_EU'], ascending=[True, True, False])
	  .groupby(['dataset', 'num_layers'], as_index=False)
	  .head(3)
	  .reset_index(drop=True)
)

# mean across all datasets per gnn_type (using the selected top-3 per dataset)
mean_results = best_results.groupby('gnn_type', as_index=False)['test_auroc_EU'].mean()

# (optional) mean per dataset and gnn_type (i.e. mean of the top-3 within each dataset)
mean_per_dataset = best_results.groupby(['dataset', 'num_layers'], as_index=False)['test_auroc_EU'].mean()

In [ ]:
mean_per_dataset

In [ ]:
# compute the percentage difference for the two different gnn_types per dataset
mean_per_dataset_pivot = mean_per_dataset.pivot(index='dataset', columns='num_layers', values='test_auroc_EU').reset_index()
mean_per_dataset_pivot['percentage_diff'] = (
	(mean_per_dataset_pivot[2] - mean_per_dataset_pivot[3]) / mean_per_dataset_pivot[3]
) * 100
mean_per_dataset_pivot